# 07 — Optical Tweezer Scaling

**Primary specification:** array scale specifies control constraints.

Notebook 07 develops the first computation branch of `scalable-platforms`: large optical tweezer arrays for neutral-atom computation. It connects array size, filling fraction, rearrangement, coherent transport, control overhead, and error-correction pathways.


## 1. Start here

Notebook 00 established the repository-level statement:

> **Coherence specifies scalable platforms.**

Notebook 07 specializes that statement to optical tweezer arrays:

\[
\text{array scale}
\rightarrow
\text{many neutral atoms}
\rightarrow
\text{filling + rearrangement}
\rightarrow
\text{coherent transport}
\rightarrow
\text{control overhead}
\rightarrow
\text{error-correction pathways}
\]

The central shift is:

> **Scaling an array means scaling control, not only increasing atom count.**


In [ ]:
from pathlib import Path
import json
import math
import zipfile

import matplotlib.pyplot as plt
import pandas as pd

NOTEBOOK_NAME = "07_optical_tweezer_scaling"
CWD = Path.cwd().resolve()

if (CWD / "notebooks").exists() or (CWD / "figures").exists():
    REPO_ROOT = CWD
elif CWD.name == "notebooks":
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / NOTEBOOK_NAME
RESULT_FIGURES = RESULTS / "figures"

for path in [NOTEBOOKS, FIGURES, RESULTS, RESULT_FIGURES]:
    path.mkdir(parents=True, exist_ok=True)

def save_png(fig, name, dpi=220):
    repo_path = FIGURES / f"{name}.png"
    result_path = RESULT_FIGURES / f"{name}.png"
    fig.savefig(repo_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(result_path, dpi=dpi, bbox_inches="tight")
    print(f"✓ Saved {repo_path}")
    print(f"✓ Saved {result_path}")
    return repo_path

print(f"Repo root: {REPO_ROOT}")
print(f"Figures:   {FIGURES}")
print(f"Results:   {RESULTS}")


## 2. Scaling variables

Notebook 07 uses a compact set of engineering variables.

\[
N_{\mathrm{atoms}} = f N_{\mathrm{sites}}
\]

and

\[
R_{\mathrm{control}}
=
\frac{N_{\mathrm{move}}T_{\mathrm{move}}}{T_{\mathrm{coh}}}
\]


In [ ]:
scaling_variables = pd.DataFrame([
    {"symbol": "N_sites", "meaning": "available optical tweezer sites", "role": "array scale"},
    {"symbol": "N_atoms", "meaning": "trapped neutral atoms", "role": "available qubits"},
    {"symbol": "f", "meaning": "filling fraction", "role": "site occupancy"},
    {"symbol": "N_move", "meaning": "atoms moved during rearrangement", "role": "transport workload"},
    {"symbol": "T_move", "meaning": "time per move operation", "role": "transport latency"},
    {"symbol": "T_coh", "meaning": "useful coherence time", "role": "coherence budget"},
    {"symbol": "R_control", "meaning": "control overhead", "role": "scaling constraint"},
])
variables_path = RESULTS / "scaling_variables.csv"
scaling_variables.to_csv(variables_path, index=False)
scaling_variables


In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 7.2))
ax.axis("off")
ax.text(0.5, 0.78, "Array Scale Specifies\nControl Constraints", ha="center", va="center", fontsize=34, weight="bold")
ax.text(0.5, 0.56, "Optical tweezer arrays scale computation by scaling\nneutral atoms, filling, transport, and control.", ha="center", va="center", fontsize=17)
ax.text(0.5, 0.36, "array scale  →  filling fraction  →  rearrangement  →  coherent transport  →  error correction", ha="center", va="center", fontsize=14, bbox=dict(boxstyle="round,pad=0.65", linewidth=1.2, facecolor="white"))
ax.text(0.5, 0.18, "large arrays  •  high filling  •  low overhead  •  long coherence", ha="center", va="center", fontsize=14)
fig.tight_layout()
hero_fig = save_png(fig, "07_hero_statement")
plt.show()
hero_fig


## 3. Optical tweezer scaling chain


In [ ]:
chain_steps = ["Optical\ntweezer sites","Loaded\natoms","Filling\nfraction","Rearrangement","Coherent\ntransport","Logical\nlayout","Error\ncorrection"]
fig, ax = plt.subplots(figsize=(13.5, 3.4))
ax.axis("off"); y = 0.5
for i, label in enumerate(chain_steps):
    ax.text(i, y, label, ha="center", va="center", fontsize=11.5,
            bbox=dict(boxstyle="round,pad=0.52", linewidth=1.2, facecolor="white"))
    if i < len(chain_steps)-1:
        ax.annotate("", xy=(i+0.68,y), xytext=(i+0.32,y), arrowprops=dict(arrowstyle="->", linewidth=1.6))
ax.set_xlim(-0.6, len(chain_steps)-0.4); ax.set_ylim(0,1)
ax.set_title("Array scale becomes useful through control", fontsize=16)
fig.tight_layout()
chain_fig = save_png(fig, "07_array_scaling_chain")
plt.show()
chain_fig


## 4. Capacity from filling fraction


In [ ]:
site_counts = [500, 1000, 2000, 4000, 6000, 8000, 12000]
fillings = [0.35, 0.50, 0.65, 0.80, 0.90]
capacity = pd.DataFrame([{"N_sites": n, "filling_fraction": f, "N_atoms": f*n} for n in site_counts for f in fillings])
capacity_path = RESULTS / "tweezer_capacity.csv"
capacity.to_csv(capacity_path, index=False)
capacity.head()


In [ ]:
fig, ax = plt.subplots(figsize=(9,5.2))
for f in fillings:
    sub = capacity[capacity["filling_fraction"] == f]
    ax.plot(sub["N_sites"], sub["N_atoms"], marker="o", label=f"f = {f:.2f}")
ax.set_xlabel("Tweezer sites"); ax.set_ylabel("Trapped atoms")
ax.set_title("Capacity scales with both site count and filling fraction")
ax.legend(title="Filling")
fig.tight_layout()
capacity_fig = save_png(fig, "07_tweezer_capacity")
plt.show()
capacity_fig


## 5. Filling fraction as an engineering lever


In [ ]:
fixed_sites = 12000
fill_grid = [i / 100 for i in range(20, 101, 5)]
filling_curve = pd.DataFrame({"filling_fraction": fill_grid, "N_atoms": [fixed_sites*f for f in fill_grid]})
filling_path = RESULTS / "filling_fraction_curve.csv"
filling_curve.to_csv(filling_path, index=False)
filling_curve.head()


In [ ]:
fig, ax = plt.subplots(figsize=(8.5,5.2))
ax.plot(filling_curve["filling_fraction"], filling_curve["N_atoms"], marker="o")
ax.set_xlabel("Filling fraction"); ax.set_ylabel("Usable atoms")
ax.set_title("At fixed site count, filling fraction sets usable scale")
ax.set_ylim(0, fixed_sites*1.05)
fig.tight_layout()
filling_fig = save_png(fig, "07_filling_fraction")
plt.show()
filling_fig


## 6. Transport overhead


In [ ]:
T_coh = 10.0
move_counts = [50, 100, 200, 500, 1000, 2000, 4000]
move_times_ms = [0.05, 0.10, 0.25, 0.50, 1.00]
overhead = pd.DataFrame([{"N_move": n, "T_move_ms": t, "T_coh_s": T_coh, "R_control": n*(t/1000.0)/T_coh} for n in move_counts for t in move_times_ms])
overhead_path = RESULTS / "transport_overhead.csv"
overhead.to_csv(overhead_path, index=False)
overhead.head()


In [ ]:
fig, ax = plt.subplots(figsize=(9,5.2))
for t in move_times_ms:
    sub = overhead[overhead["T_move_ms"] == t]
    ax.plot(sub["N_move"], sub["R_control"], marker="o", label=f"{t:g} ms/move")
ax.axhline(0.1, linestyle="--", linewidth=1.2)
ax.text(max(move_counts)*0.72, 0.105, "10% coherence budget", fontsize=10)
ax.set_xlabel("Moved atoms"); ax.set_ylabel("Control overhead")
ax.set_title("Transport overhead competes with coherence budget")
ax.legend(title="Move time")
fig.tight_layout()
overhead_fig = save_png(fig, "07_transport_overhead")
plt.show()
overhead_fig


## 7. Scaling constraints map


In [ ]:
fig, ax = plt.subplots(figsize=(12,6.2))
ax.axis("off")
ax.text(0.5, 0.94, "Scaling constraints for optical tweezer platforms", ha="center", va="center", fontsize=18, weight="bold")
ax.text(0.5, 0.86, "Array size becomes useful when control preserves coherence.", ha="center", va="center", fontsize=12)
ax.text(0.5, 0.72, "USEFUL SCALE", ha="center", va="center", fontsize=15, weight="bold",
        bbox=dict(boxstyle="round,pad=0.65", linewidth=1.5, facecolor="white"))
constraint_boxes = [
    (0.22,0.53,"Large arrays","many sites"), (0.42,0.53,"High filling","usable atoms"),
    (0.62,0.53,"Low overhead","fast rearrangement"), (0.82,0.53,"Long coherence","control budget"),
    (0.32,0.30,"Transport","move without loss"), (0.52,0.30,"Addressing","low crosstalk"),
    (0.72,0.30,"Logical layout","QEC pathway"),
]
for x,y,title,subtitle in constraint_boxes:
    ax.text(x,y,f"{title}\n{subtitle}",ha="center",va="center",fontsize=11,
            bbox=dict(boxstyle="round,pad=0.45", linewidth=1.1, facecolor="#F7F8FA"))
    ax.annotate("", xy=(0.5,0.66), xytext=(x,y+0.055), arrowprops=dict(arrowstyle="->", linewidth=1.1))
ax.text(0.5, 0.12, "array scale  →  control constraints  →  coherent layouts  →  error correction", ha="center", va="center", fontsize=13)
fig.tight_layout()
constraints_fig = save_png(fig, "07_scaling_constraints")
plt.show()
constraints_fig


## 8. Notebook summary


In [ ]:
summary = pd.DataFrame([
    {"layer": "array scale", "specification": "more sites expand possible layouts"},
    {"layer": "filling fraction", "specification": "site occupancy controls usable atom count"},
    {"layer": "rearrangement", "specification": "sorting converts random loading into useful layouts"},
    {"layer": "transport", "specification": "movement must preserve state information"},
    {"layer": "control overhead", "specification": "transport must fit inside the coherence budget"},
    {"layer": "error correction", "specification": "useful scale points toward logical layouts"},
])
summary_path = RESULTS / "notebook_07_summary.csv"
summary.to_csv(summary_path, index=False)
summary


In [ ]:
fig, ax = plt.subplots(figsize=(12.5,5.8))
ax.axis("off")
ax.text(0.5,0.93,"Notebook 07 summary",ha="center",va="center",fontsize=18,weight="bold")
ax.text(0.5,0.85,"Array scale specifies control constraints.",ha="center",va="center",fontsize=14)
steps=[("Sites","array\nscale"),("Atoms","filling\nfraction"),("Moves","rearrange"),("Transport","preserve\ncoherence"),("Layout","logical\nstructure"),("QEC","fault-tolerant\npathway")]
for i,(top,bottom) in enumerate(steps):
    x = 0.10 + i*0.16
    ax.text(x,0.58,top,ha="center",va="center",fontsize=13,weight="bold",
            bbox=dict(boxstyle="round,pad=0.5",linewidth=1.2,facecolor="white"))
    ax.text(x,0.34,bottom,ha="center",va="center",fontsize=11)
    if i < len(steps)-1:
        ax.annotate("", xy=(x+0.105,0.58), xytext=(x+0.055,0.58), arrowprops=dict(arrowstyle="->",linewidth=1.3))
ax.text(0.5,0.12,"Large arrays become platforms when control remains coherent.",ha="center",va="center",fontsize=13)
fig.tight_layout()
summary_fig = save_png(fig, "07_notebook_summary")
plt.show()
summary_fig


## 9. Save notebook manifest


In [ ]:
notebook_manifest = {
    "notebook": "07_optical_tweezer_scaling.ipynb",
    "title": "Optical Tweezer Scaling",
    "primary_specification": "array scale specifies control constraints",
    "statement": "Scaling an array means scaling control, not only increasing atom count.",
    "outputs": {
        "scaling_variables": str(variables_path.relative_to(REPO_ROOT)),
        "capacity_csv": str(capacity_path.relative_to(REPO_ROOT)),
        "filling_csv": str(filling_path.relative_to(REPO_ROOT)),
        "overhead_csv": str(overhead_path.relative_to(REPO_ROOT)),
        "summary_csv": str(summary_path.relative_to(REPO_ROOT)),
        "hero_figure": str(hero_fig.relative_to(REPO_ROOT)),
        "chain_figure": str(chain_fig.relative_to(REPO_ROOT)),
        "capacity_figure": str(capacity_fig.relative_to(REPO_ROOT)),
        "filling_figure": str(filling_fig.relative_to(REPO_ROOT)),
        "overhead_figure": str(overhead_fig.relative_to(REPO_ROOT)),
        "constraints_figure": str(constraints_fig.relative_to(REPO_ROOT)),
        "summary_figure": str(summary_fig.relative_to(REPO_ROOT)),
    },
}
manifest_path = RESULTS / "07_optical_tweezer_scaling_manifest.json"
manifest_path.write_text(json.dumps(notebook_manifest, indent=2), encoding="utf-8")
notebook_manifest


## 10. Download artifacts

Run the final cell to package the notebook outputs for download.


In [ ]:
from IPython.display import FileLink, display
zip_path = RESULTS / "07_optical_tweezer_scaling_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in [FIGURES, RESULTS]:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if path.is_file() and path.resolve() != zip_path.resolve():
                    try:
                        archive_name = path.relative_to(REPO_ROOT)
                    except ValueError:
                        archive_name = path.name
                    zf.write(path, archive_name)
        elif item.exists():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)
print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
